# 42 — Multi Site Comparison

Builds a simple target vs control comparison from the cross-reference table.

In [ ]:
import os, json, hashlib, platform, sys, math, time
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "42_multi_site_comparison"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

xref, xmeta = safe_read_csv("outputs/41_cross_reference_existing_outputs/cross_reference_daily.csv")
if xref.empty:
    raise FileNotFoundError("Run notebook 41 first.")

xref["date"] = pd.to_datetime(xref["date"], errors="coerce")

value_col = "site_score"
if value_col not in xref.columns:
    xref[value_col] = 0

target = xref[xref["role"] == "target"].groupby("date", as_index=False)[value_col].mean().rename(columns={value_col: "target_score"})
control = xref[xref["role"] == "control"].groupby("date", as_index=False)[value_col].mean().rename(columns={value_col: "control_score"})
daily = target.merge(control, on="date", how="outer").sort_values("date").fillna(0)
daily["target_minus_control"] = daily["target_score"] - daily["control_score"]
daily["abs_difference"] = daily["target_minus_control"].abs()

extra_cols = [c for c in ["article_count","ground_rows"] if c in xref.columns]
if extra_cols:
    extras = xref.groupby("date", as_index=False)[extra_cols].max()
    daily = daily.merge(extras, on="date", how="left")

site_summary = xref.groupby(["site_id","site_name","role"], as_index=False).agg(
    mean_site_score=("site_score","mean"),
    max_site_score=("site_score","max"),
    obs_days=("date","nunique")
)

daily.to_csv(out / "daily_comparison.csv", index=False)
site_summary.to_csv(out / "site_summary.csv", index=False)
display(site_summary)
display(daily.head(20))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily["date"], daily["target_minus_control"], marker="o")
ax.set_title("Target minus control score")
ax.set_xlabel("Date")
ax.set_ylabel("Difference")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(out / "target_minus_control_plot.png", dpi=150, bbox_inches="tight")

manifest = manifest_base(step, [CONFIGS / "run.yml"])
for p in [out / "daily_comparison.csv", out / "site_summary.csv", out / "target_minus_control_plot.png"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["inputs"] = {"cross_reference": xmeta}
write_json(out / "manifest.json", manifest)
print("Wrote:", out)